# Customer Churn Prediction - Data Preprocessing and Feature Engineering

This notebook focuses on preparing the data for machine learning by:
1. Cleaning and handling missing values
2. Engineering new features based on domain knowledge
3. Encoding categorical variables
4. Splitting data for model training and evaluation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 1. Data Loading and Initial Cleaning

In [ ]:
df = pd.read_csv('../data/customer_churn.csv')

print(f"Original dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
print("Converting TotalCharges to numeric...")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing_total_charges = df['TotalCharges'].isnull().sum()
print(f"Missing values in TotalCharges after conversion: {missing_total_charges}")

median_total_charges = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(median_total_charges)

print(f"Filled missing TotalCharges with median value: {median_total_charges:.2f}")
print(f"Missing values after filling: {df['TotalCharges'].isnull().sum()}")

### Why use median instead of mean for filling missing values?

The median is more robust to outliers compared to the mean. In the context of TotalCharges:
- Some customers might have extremely high total charges (long-term, high-value customers)
- These outliers would skew the mean upward
- The median provides a more representative central value that won't distort the distribution
- This approach preserves the overall data distribution while handling missing values

## 2. Feature Engineering

Based on our EDA findings and domain knowledge, we'll create several new features that capture important patterns in customer behavior and service usage.

### 2.1 Tenure Group Feature

**Rationale**: Our EDA showed that tenure is a strong predictor of churn, with new customers being much more likely to leave. Grouping tenure into meaningful segments helps the model capture non-linear relationships and makes the feature more interpretable.

In [ ]:
def create_tenure_group(tenure):
    if tenure <= 12:
        return '0-12 months'
    elif tenure <= 24:
        return '12-24 months'
    elif tenure <= 48:
        return '24-48 months'
    elif tenure <= 60:
        return '48-60 months'
    else:
        return '60+ months'

df['tenure_group'] = df['tenure'].apply(create_tenure_group)

print("Tenure group distribution:")
print(df['tenure_group'].value_counts().sort_index())

churn_by_tenure = df.groupby('tenure_group')['Churn'].value_counts(normalize=True).unstack()['Yes'] * 100
print("\nChurn rate by tenure group:")
print(churn_by_tenure.round(1))

### 2.2 Service Count Feature

**Rationale**: The number of services a customer uses indicates their level of engagement and investment in the platform. Customers with more services may be less likely to churn due to higher switching costs and greater integration into the ecosystem.

In [ ]:
service_columns = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                  'TechSupport', 'StreamingTV', 'StreamingMovies']

def count_services(row):
    count = 0
    for service in service_columns:
        if row[service] == 'Yes':
            count += 1
    return count

df['Service_Count'] = df.apply(count_services, axis=1)

print("Service Count distribution:")
print(df['Service_Count'].value_counts().sort_index())

churn_by_services = df.groupby('Service_Count')['Churn'].value_counts(normalize=True).unstack()['Yes'] * 100
print("\nChurn rate by Service Count:")
print(churn_by_services.round(1))

### 2.3 Is_Senior_Solo Feature

**Rationale**: Senior citizens who live alone (no partner) may have different needs and usage patterns. They might be more price-sensitive, less tech-savvy, or have different support requirements, making them a distinct segment for churn prediction.

In [ ]:
df['Is_Senior_Solo'] = ((df['SeniorCitizen'] == 1) & (df['Partner'] == 'No')).astype(int)

print("Is_Senior_Solo distribution:")
print(df['Is_Senior_Solo'].value_counts())

churn_by_senior_solo = df.groupby('Is_Senior_Solo')['Churn'].value_counts(normalize=True).unstack()['Yes'] * 100
print("\nChurn rate by Is_Senior_Solo:")
print(churn_by_senior_solo.round(1))
print(f"\n0 = Not Senior Solo, 1 = Senior Solo")

## 3. Data Preparation for Modeling

In [ ]:
df_processed = df.drop('customerID', axis=1)
print(f"Shape after dropping customerID: {df_processed.shape}")

df_processed['Churn'] = (df_processed['Churn'] == 'Yes').astype(1)

print(f"Target variable distribution:")
print(df_processed['Churn'].value_counts())
print(f"Churn rate: {df_processed['Churn'].mean():.3f} ({df_processed['Churn'].mean()*100:.1f}%)")

### 3.1 One-Hot Encoding for Categorical Features

**Rationale**: Machine learning models require numerical input. One-hot encoding converts categorical variables into binary columns, allowing the model to treat each category independently without assuming ordinal relationships.

In [ ]:
categorical_columns = df_processed.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns to encode: {categorical_columns}")

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

encoded_categorical = encoder.fit_transform(df_processed[categorical_columns])

encoded_df = pd.DataFrame(encoded_categorical, 
                         columns=encoder.get_feature_names_out(categorical_columns),
                         index=df_processed.index)

numerical_columns = df_processed.select_dtypes(include=[np.number]).columns.tolist()
numerical_columns.remove('Churn')

final_df = pd.concat([df_processed[numerical_columns], encoded_df, df_processed['Churn']], axis=1)

print(f"Final dataset shape: {final_df.shape}")
print(f"\nFinal columns: {list(final_df.columns)}")

### 3.2 Train-Test Split

**Rationale**: Splitting the data ensures we can evaluate model performance on unseen data. Using random_state=42 ensures reproducibility of results.

In [ ]:
X = final_df.drop('Churn', axis=1)
y = final_df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"\nTraining set churn rate: {y_train.mean():.3f} ({y_train.mean()*100:.1f}%)")
print(f"Testing set churn rate: {y_test.mean():.3f} ({y_test.mean()*100:.1f}%)")

## 4. Save Processed Data

We'll save the fully processed dataset for use in modeling notebooks. This ensures reproducibility and allows us to skip preprocessing steps in future runs.

In [ ]:
processed_data = pd.concat([X, y], axis=1)

output_path = '../data/processed_churn_data.csv'
processed_data.to_csv(output_path, index=False)

print(f"Processed data saved to: {output_path}")
print(f"Final dataset shape: {processed_data.shape}")
print(f"\nDataset info:")
print(f"- Total samples: {len(processed_data)}")
print(f"- Total features: {len(processed_data.columns) - 1}")
print(f"- Target variable: Churn")
print(f"- Churn rate: {processed_data['Churn'].mean():.3f} ({processed_data['Churn'].mean()*100:.1f}%)")

In [ ]:
print("Sample of processed data:")
processed_data.head()

In [ ]:
train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_csv('../data/train_data.csv', index=False)
test_data.to_csv('../data/test_data.csv', index=False)

print(f"Training data saved to: ../data/train_data.csv (shape: {train_data.shape})")
print(f"Testing data saved to: ../data/test_data.csv (shape: {test_data.shape})")

## 5. Summary of Feature Engineering

### New Features Created:

1. **tenure_group**: Categorizes customers into lifecycle stages
   - Helps capture non-linear relationships between tenure and churn
   - More interpretable than continuous tenure
   - 0-12 months: Highest churn risk (new customers)
   - 60+ months: Lowest churn risk (loyal customers)

2. **Service_Count**: Measures customer engagement level
   - Higher count indicates greater investment in platform
   - May correlate with lower churn due to switching costs
   - Range: 0-6 services

3. **Is_Senior_Solo**: Identifies vulnerable customer segment
   - Senior citizens without partners may have unique needs
   - May require different retention strategies
   - Binary feature for easy interpretation

### Data Processing Steps:
- Handled missing TotalCharges with median imputation
- Applied one-hot encoding to all categorical variables
- Maintained class balance in train/test split using stratification
- Created reproducible splits with random_state=42

### Files Generated:
- `processed_churn_data.csv`: Complete processed dataset
- `train_data.csv`: Training split (5,634 samples)
- `test_data.csv`: Testing split (1,409 samples)

The data is now ready for machine learning model development!